# CFTC TFF 2011/2026 日期列别名核对

只读已归档官方ZIP，确认证券代码209742与两个原始报告日期列名，不增加HTTP请求。

In [1]:
import csv,io,json,hashlib,re,zipfile
from pathlib import Path
from datetime import datetime
root=Path.cwd();card=root/'research/stage1_cftc_nq_tff_source_probe_20261023';plan=json.loads((card/'mechanical_schema_alias_plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
raw=root/'data/snapshots/stage1_cftc_nq_tff_source_probe_20261023';assert plan['frozen'] and sha(card/'repair_result.json')==plan['prior_repair_result_sha256']
prior=json.loads((card/'repair_result.json').read_text());assert prior['repair_NAS_archive_sha256']==plan['NAS_archive_sha256'] and prior['NAS_remote_and_scoped_restore_verified'] and prior['cumulative_GET_attempts']==2
checks=[]
for year in (2011,2026):
 p=raw/f'fut_fin_txt_{year}.zip';assert sha(p)==plan[f'raw_{year}_sha256']
 with zipfile.ZipFile(p) as z:
  members=[x for x in z.namelist() if x.lower().endswith(('.txt','.csv'))];assert len(members)==1;content=z.read(members[0])
 try:decoded=content.decode('utf-8-sig')
 except UnicodeDecodeError:decoded=content.decode('cp1252')
 reader=csv.DictReader(io.StringIO(decoded));columns=reader.fieldnames;rows=list(reader);assert columns and len(rows)>200
 market='Market_and_Exchange_Names';code='CFTC_Contract_Market_Code';oi='Open_Interest_All';long='Lev_Money_Positions_Long_All';short='Lev_Money_Positions_Short_All';dates=[x for x in plan['exact_date_header_aliases'] if x in columns];assert len(dates)==1 and all(x in columns for x in (market,code,oi,long,short))
 report=dates[0];nq=[x for x in rows if x[code].strip()==plan['contract_code'] and all(y in x[market].upper() for y in ('NASDAQ','MINI','CHICAGO MERCANTILE'))];assert len(nq)>=20
 parsed=[datetime.strptime(x[report].strip(),'%Y-%m-%d').date() for x in nq];assert len(set(parsed))==len(parsed) and min(parsed).year==max(parsed).year==year
 num=lambda x:float(x.replace(',','').strip());assert all(num(x[oi])>0 and num(x[long])>=0 and num(x[short])>=0 for x in nq)
 checks.append({'year':year,'raw_sha256':sha(p),'contract_code':plan['contract_code'],'contract_names':sorted({x[market].strip() for x in nq}),'report_date_header':report,'NQ_report_rows':len(nq),'first_report_date':str(min(parsed)),'last_report_date':str(max(parsed)),'Tuesday_report_share':sum(d.weekday()==1 for d in parsed)/len(parsed),'market_outcome_reads':0})
result={'schema_version':1,'status':'official_CFTC_TFF_Emini_NQ_2011_and_2026_identity_passed_release_clock_pending','plan_sha256':sha(card/'mechanical_schema_alias_plan.json'),'cumulative_external_GETs':2,'source_NAS_archive_sha256':plan['NAS_archive_sha256'],'source_NAS_remote_and_scoped_restore_verified':True,'checks':checks,'report_date_is_not_release_date':True,'historical_dissemination_time_qualified':False,'model_fits':0,'native_accounts':0,'market_outcome_reads':0};(card/'schema_alias_result.json').write_text(json.dumps(result,indent=2)+'\n');print(result)


{'schema_version': 1, 'status': 'official_CFTC_TFF_Emini_NQ_2011_and_2026_identity_passed_release_clock_pending', 'plan_sha256': 'db917bde02d2f07b6ffb8ec406b711eef959acf56b3589a476f4eb488aa28db2', 'cumulative_external_GETs': 2, 'source_NAS_archive_sha256': 'e23617260a78790f41840f086d67f1e9ddd790ff42b295de29dd8f29376f978c', 'source_NAS_remote_and_scoped_restore_verified': True, 'checks': [{'year': 2011, 'raw_sha256': '583a93474847f1070b213e2b12cfa73c48d9ba05f24f7d001a338de125468b57', 'contract_code': '209742', 'contract_names': ['NASDAQ-100 STOCK INDEX (MINI) - CHICAGO MERCANTILE EXCHANGE'], 'report_date_header': 'Report_Date_as_MM_DD_YYYY', 'NQ_report_rows': 52, 'first_report_date': '2011-01-04', 'last_report_date': '2011-12-27', 'Tuesday_report_share': 1.0, 'market_outcome_reads': 0}, {'year': 2026, 'raw_sha256': '6722fc377087c9380d7504345ac7145a891185ea7d57c9ad3f184b29da43aa57', 'contract_code': '209742', 'contract_names': ['NASDAQ MINI - CHICAGO MERCANTILE EXCHANGE'], 'report_date_h

## 中文结论

最终本地别名解析通过：2011年52条、2026年41条，均为合约209742；两次 GET 用尽且 NAS 指定成员恢复通过。发布时间和首发版本仍未合格，未建模或回测。

完整科学决定见 `conclusion.md`。